In [ ]:
import pandas as pd

df = pd.read_csv("all-data.csv", encoding="latin-1")
df.columns = ['sentiment', 'news']

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

df.head()

In [ ]:
import pandas as pd

df = pd.read_csv("all-data.csv", encoding="latin-1")
df.columns = ['sentiment', 'news']

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% training, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["sentiment"]
)

# Second split: 10% validation, 10% testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["sentiment"]
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

In [ ]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Datasets saved successfully!")

Loading Qwen model

In [ ]:

import torch
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded successfully!")
print("Device:", model.device)


Initial prompts

In [ ]:

# Load test dataset if not already available
test_df = pd.read_csv("test.csv")

LABELS = ["positive", "negative", "neutral"]

def create_prompt(text):
    return f"""You are a financial news sentiment classifier.

Classify the sentiment of the following financial news.

News: {text}

Choose exactly one label:
positive, negative, or neutral.

Sentiment:"""

print(create_prompt(test_df.iloc[0]["text"]))


Original model Eval

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import re

def extract_sentiment(output):
    output = output.lower().strip()

    # Find the first valid sentiment label
    match = re.search(r"\b(positive|negative|neutral)\b", output)

    if match:
        return match.group(1)

    return "invalid"


def predict_batch(texts, batch_size=8):
    predictions = []

    model.eval()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        prompts = [create_prompt(text) for text in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        generated = outputs[:, inputs["input_ids"].shape[1]:]

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )

        predictions.extend(
            [extract_sentiment(output) for output in decoded]
        )

    return predictions


# Ground truth
y_true = test_df["sentiment"].str.lower().tolist()

# Base model predictions
base_predictions = predict_batch(test_df["text"].tolist())

# Evaluation
print("Base Model Accuracy:",
      accuracy_score(y_true, base_predictions))

print("Base Model Macro F1:",
      f1_score(y_true, base_predictions,
               labels=LABELS,
               average="macro",
               zero_division=0))

print("\nClassification Report:")
print(classification_report(
    y_true,
    base_predictions,
    labels=LABELS,
    zero_division=0
))


## Now Configuring LoRA

| Parameter | Value | Purpose |
| --- | --- | --- |
| **Rank ($r$)** | 8 | Controls adapter capacity |
| **Alpha** | 16 | Scaling factor |
| **Dropout** | 0.05 | Helps reduce overfitting |
| **Target modules** | `q_proj`, `v_proj` | Adapt attention projections |
| **Learning rate** | `2e-4` | Controls parameter updates |
| **Epochs** | 3 | Number of training passes |

In [ ]:

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(model, lora_config)

lora_model.print_trainable_parameters()


In [ ]:

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(model, lora_config)

lora_model.print_trainable_parameters()


Preparing training data

In [ ]:

from datasets import Dataset

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")

def prepare_dataset(df):
    examples = []

    for _, row in df.iterrows():
        prompt = create_prompt(row["news"])
        answer = row["sentiment"].strip().lower()

        full_text = prompt + " " + answer + tokenizer.eos_token

        tokenized_full = tokenizer(
            full_text,
            truncation=True,
            max_length=256
        )

        tokenized_prompt = tokenizer(
            prompt,
            truncation=True,
            max_length=256
        )

        input_ids = tokenized_full["input_ids"]
        prompt_length = len(tokenized_prompt["input_ids"])

        labels = input_ids.copy()

        # Ignore prompt tokens during loss calculation
        labels[:prompt_length] = [-100] * min(
            prompt_length, len(labels)
        )

        examples.append({
            "input_ids": input_ids,
            "attention_mask": tokenized_full["attention_mask"],
            "labels": labels
        })

    return Dataset.from_list(examples)


train_dataset = prepare_dataset(train_df)
val_dataset = prepare_dataset(val_df)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("\nSample training example:")
print(train_dataset[0])


using hugging face trainer

In [ ]:

from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./qwen-lora-financial",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

print("Trainer configured successfully!")


In [ ]:
trainer.train()